# Análisis de Correlación Temporal
## Entendiendo la Autocorrelación: ACF y PACF

> **Fase 1 — Video 4** | Análisis Exploratorio de Datos  
> Dataset: Histórico de Ventas Sintético

---

### ¿Por qué necesitamos ACF y PACF?

En el Video 3 usamos Ljung-Box para validar si el residual era ruido blanco.  
Ese test responde **sí o no**. ACF y PACF responden **cuánto y en qué rezagos** — son el mapa completo.

Su uso principal: **identificar los parámetros p, d, q de un modelo ARIMA** antes de modelar.  
Pero también revelan estructura oculta en la serie que ningún otro gráfico muestra.

| Herramienta | Mide | Para qué sirve |
|---|---|---|
| **ACF** | Correlación entre `y_t` y `y_{t-k}` | Identificar `q` (MA) y estacionalidad |
| **PACF** | Correlación directa entre `y_t` y `y_{t-k}` eliminando intermedios | Identificar `p` (AR) |

### Ruta del notebook
1. Librerías y carga de datos
2. Intuición: ¿Qué mide realmente la autocorrelación?
3. ACF — Leyendo el correlograma
4. PACF — La correlación directa
5. ACF + PACF sobre la serie diferenciada
6. Patrones estacionales en ACF y PACF
7. Tabla de decisión: identificando p, d, q
8. Conclusiones y próximos pasos

---
## 1. Librerías y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from matplotlib.patches import FancyArrowPatch
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0',
    'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563EB','#DC2626','#16A34A','#7C3AED','#EA580C'
money_fmt = FuncFormatter(lambda x, _: f'${x/1e6:.1f}M')
print('Librerías cargadas')

In [ ]:
def find_csv(filename='sales_history.csv', max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")

csv_path = find_csv()
df_raw   = pd.read_csv(csv_path, nrows=2)
date_col = next(c for c in df_raw.columns if 'date' in c.strip().lower())
df = pd.read_csv(csv_path, parse_dates=[date_col])
df.rename(columns={date_col: 'date'}, inplace=True)
df.sort_values('date', inplace=True)

# Serie semanal — misma que Videos 2 y 3
serie = df.groupby('date')['revenue'].sum().resample('W-MON').sum()
serie.name = 'revenue'

# Serie desestacionalizada usando el pipeline del Video 3
serie_log = np.log1p(serie)
res_stl   = STL(serie_log, period=52, seasonal=53, robust=True).fit()
deseason  = serie / res_stl.seasonal.apply(np.exp)   # multiplicativo: dividir por factor
deseason.name = 'revenue_desestacionalizado'

print(f'✅ Serie cargada: {len(serie)} semanas | {serie.index.min().date()} → {serie.index.max().date()}')
print('   Serie desestacionalizada lista para análisis.')

---
## 2. Intuición: ¿Qué mide realmente la autocorrelación?

La autocorrelación mide qué tan parecida es una serie **consigo misma desplazada en el tiempo**.

- **Rezago 1 (lag 1):** ¿Las ventas de esta semana predicen las de la semana siguiente?
- **Rezago 52 (lag 52):** ¿Las ventas de esta semana predicen las del mismo período del año pasado?

Un valor cercano a **+1** → fuerte correlación positiva  
Un valor cercano a **-1** → fuerte correlación negativa  
Un valor cercano a **0** → sin relación → ruido en ese rezago

In [ ]:
# Visualizamos la serie vs ella misma desplazada en 1, 4 y 52 semanas
lags_demo = [1, 4, 52]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, lag in zip(axes, lags_demo):
    y      = serie.values[lag:]
    y_lag  = serie.values[:-lag]
    corr   = np.corrcoef(y, y_lag)[0, 1]
    ax.scatter(y_lag, y, alpha=0.4, color=BLUE, s=15)
    # Línea de tendencia
    m, b = np.polyfit(y_lag, y, 1)
    x_line = np.linspace(y_lag.min(), y_lag.max(), 100)
    ax.plot(x_line, m * x_line + b, color=RED, linewidth=2)
    ax.xaxis.set_major_formatter(money_fmt)
    ax.yaxis.set_major_formatter(money_fmt)
    ax.set_title(f'Lag {lag} — r = {corr:.3f}', fontsize=12, fontweight='bold')
    ax.set_xlabel(f'Ventas semana t−{lag}')
    ax.set_ylabel('Ventas semana t')

fig.suptitle('Scatter de Autocorrelación: ¿La serie se predice a sí misma?',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Interpretación:')
for lag in lags_demo:
    y, y_lag = serie.values[lag:], serie.values[:-lag]
    r = np.corrcoef(y, y_lag)[0, 1]
    strength = 'Fuerte' if abs(r) > 0.7 else 'Moderada' if abs(r) > 0.4 else 'Débil'
    print(f'  Lag {lag:>2}: r = {r:.3f} → {strength}')

---
## 3. ACF — Leyendo el correlograma

El correlograma ACF grafica la correlación para **todos los rezagos a la vez**.  
La banda sombreada es el **intervalo de confianza al 95%**: barras que la cruzan son estadísticamente significativas.

### Patrones clave a reconocer

| Patrón en ACF | Diagnóstico |
|---|---|
| Decaimiento **muy lento** (casi lineal) | Serie con **tendencia** — no estacionaria |
| Decaimiento **exponencial rápido** | Serie **estacionaria** — proceso AR |
| **Corte abrupto** en lag k | Proceso **MA(q)** — q es el último lag significativo |
| Picos periódicos cada **s** lags | **Estacionalidad** de período s |

In [ ]:
LAGS = 60

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# ACF de la serie ORIGINAL
plot_acf(serie, lags=LAGS, ax=axes[0], color=BLUE, alpha=0.05)
axes[0].set_title(
    'ACF — Serie Original\n'
    '→ Decaimiento lento = tendencia presente | Picos periódicos = estacionalidad',
    fontsize=12, fontweight='bold')
axes[0].set_xlabel('Rezago (semanas)')
axes[0].set_ylabel('Autocorrelación')
axes[0].axhline(0, color='black', linewidth=0.8)

# ACF de la serie DESESTACIONALIZADA
plot_acf(deseason, lags=LAGS, ax=axes[1], color=GREEN, alpha=0.05)
axes[1].set_title(
    'ACF — Serie Desestacionalizada\n'
    '→ Sin picos periódicos: la estacionalidad fue removida en el Video 3',
    fontsize=12, fontweight='bold')
axes[1].set_xlabel('Rezago (semanas)')
axes[1].set_ylabel('Autocorrelación')
axes[1].axhline(0, color='black', linewidth=0.8)

fig.suptitle('ACF: Original vs Desestacionalizada', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Valores numéricos de la ACF
acf_vals  = acf(serie, nlags=LAGS, fft=True)
conf_int  = 1.96 / np.sqrt(len(serie))
sig_lags  = [i for i, v in enumerate(acf_vals) if abs(v) > conf_int and i > 0]
print(f'Serie original — rezagos significativos (|r| > {conf_int:.3f}): {sig_lags[:15]}...')
print(f'→ {len(sig_lags)} rezagos significativos de {LAGS} analizados')

---
## 4. PACF — La correlación directa

La **correlación parcial** elimina el efecto de los rezagos intermedios.

**Analogía:** si las ventas del lunes predicen las del miércoles, y las del martes también predicen las del miércoles, la PACF mide la contribución **directa** del lunes — sin contar que pasó el martes por en medio.

### Patrones clave a reconocer

| Patrón en PACF | Diagnóstico |
|---|---|
| **Corte abrupto** en lag p | Proceso **AR(p)** — p es el último lag significativo |
| Decaimiento **exponencial** | Proceso **MA** |
| Decaimiento **sinusoidal** | Proceso AR con raíces complejas |

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

plot_pacf(serie, lags=LAGS, ax=axes[0], color=PURPLE, alpha=0.05, method='ywm')
axes[0].set_title(
    'PACF — Serie Original\n'
    '→ Primer rezago dominante = fuerte dependencia del valor anterior',
    fontsize=12, fontweight='bold')
axes[0].set_xlabel('Rezago (semanas)')
axes[0].set_ylabel('Autocorrelación Parcial')
axes[0].axhline(0, color='black', linewidth=0.8)

plot_pacf(deseason, lags=LAGS, ax=axes[1], color=ORANGE, alpha=0.05, method='ywm')
axes[1].set_title(
    'PACF — Serie Desestacionalizada\n'
    '→ Estructura más limpia para identificar el orden AR',
    fontsize=12, fontweight='bold')
axes[1].set_xlabel('Rezago (semanas)')
axes[1].set_ylabel('Autocorrelación Parcial')
axes[1].axhline(0, color='black', linewidth=0.8)

fig.suptitle('PACF: Original vs Desestacionalizada', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

pacf_vals = pacf(serie, nlags=LAGS, method='ywm')
sig_pacf  = [i for i, v in enumerate(pacf_vals) if abs(v) > conf_int and i > 0]
print(f'Serie original — rezagos PACF significativos: {sig_pacf[:10]}...')

---
## 5. ACF + PACF sobre la serie diferenciada

En el Video 2 establecimos que la serie necesita **d=1** para ser estacionaria.  
Los modelos ARIMA trabajan sobre la serie diferenciada — por eso identificamos p y q **sobre Δ1**, no sobre la serie original.

> **Regla práctica:**  
> Siempre analiza ACF y PACF **después** de diferenciar (`d` veces) y **después** de desestacionalizar.

In [ ]:
diff1 = deseason.diff().dropna()

fig = plt.figure(figsize=(14, 12))
gs  = gridspec.GridSpec(3, 2, hspace=0.55, wspace=0.35)

# Fila 1: la serie diferenciada
ax_serie = fig.add_subplot(gs[0, :])
ax_serie.plot(diff1.index, diff1.values, color=GREEN, linewidth=1, alpha=0.85)
ax_serie.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax_serie.yaxis.set_major_formatter(money_fmt)
ax_serie.set_title('Serie Desestacionalizada + Primera Diferencia (Δ1)',
                   fontsize=12, fontweight='bold')

# Fila 2: ACF y PACF de la diferenciada
ax_acf  = fig.add_subplot(gs[1, 0])
ax_pacf = fig.add_subplot(gs[1, 1])
plot_acf( diff1, lags=40, ax=ax_acf,  color=BLUE,   alpha=0.05)
plot_pacf(diff1, lags=40, ax=ax_pacf, color=PURPLE, alpha=0.05, method='ywm')
ax_acf.set_title('ACF — Δ1 Desestacionalizada', fontsize=11, fontweight='bold')
ax_pacf.set_title('PACF — Δ1 Desestacionalizada', fontsize=11, fontweight='bold')
ax_acf.set_xlabel('Rezago')
ax_pacf.set_xlabel('Rezago')

# Fila 3: comparación visual ACF original vs diferenciada
acf_orig_vals = acf(deseason,  nlags=40, fft=True)
acf_diff_vals = acf(diff1,     nlags=40, fft=True)
lags_x = np.arange(len(acf_orig_vals))

ax_comp = fig.add_subplot(gs[2, :])
ax_comp.plot(lags_x, acf_orig_vals, color=BLUE,  linewidth=2, label='ACF desestacionalizada (original)')
ax_comp.plot(lags_x, acf_diff_vals, color=GREEN, linewidth=2, label='ACF desestacionalizada (Δ1)')
ax_comp.axhline(0,    color='black', linewidth=0.8)
ax_comp.axhline( conf_int, color=RED, linewidth=1, linestyle='--', alpha=0.6, label='IC 95%')
ax_comp.axhline(-conf_int, color=RED, linewidth=1, linestyle='--', alpha=0.6)
ax_comp.set_title('ACF: Antes vs Después de Diferenciar', fontsize=11, fontweight='bold')
ax_comp.set_xlabel('Rezago')
ax_comp.set_ylabel('Autocorrelación')
ax_comp.legend()

fig.suptitle('Análisis de la Serie Diferenciada — Preparación para ARIMA',
             fontsize=14, fontweight='bold')
plt.show()

---
## 6. Patrones estacionales en ACF y PACF

La estacionalidad tiene una **firma visual muy clara** en el correlograma:  
picos significativos en los **múltiplos del período** (para series semanales: lags 52, 104, ...).

Comparamos aquí la ACF de la serie **con** y **sin** estacionalidad para que la diferencia sea obvia.

In [ ]:
LAGS_S = 110   # más de 2 períodos para ver los harmónicos

acf_con    = acf(serie,    nlags=LAGS_S, fft=True)
acf_sin    = acf(deseason, nlags=LAGS_S, fft=True)
lags_s     = np.arange(len(acf_con))
conf_s     = 1.96 / np.sqrt(len(serie))

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

for ax, vals, label, color, title_suffix in [
    (axes[0], acf_con, 'Con estacionalidad',    BLUE,  'CON Estacionalidad — Picos en lag 52 y 104'),
    (axes[1], acf_sin, 'Sin estacionalidad',    GREEN, 'SIN Estacionalidad — Picos desaparecen'),
]:
    markercolors = [
        RED if (i % 52 == 0 and i > 0) else color
        for i in range(len(vals))
    ]
    ax.vlines(lags_s, 0, vals, colors=markercolors, linewidth=1.5, alpha=0.8)
    ax.scatter(lags_s, vals, color=markercolors, s=15, zorder=3)
    ax.axhline(0,       color='black', linewidth=0.8)
    ax.axhline( conf_s, color=RED,   linewidth=1, linestyle='--', alpha=0.5)
    ax.axhline(-conf_s, color=RED,   linewidth=1, linestyle='--', alpha=0.5)
    # Marcadores en los múltiplos del período
    for mult in [52, 104]:
        if mult <= LAGS_S:
            ax.axvline(mult, color=RED, linewidth=1.5, linestyle=':', alpha=0.7)
            ax.text(mult + 0.5, ax.get_ylim()[1] * 0.85,
                    f'lag {mult}', color=RED, fontsize=9, fontweight='bold')
    ax.set_title(f'ACF — {title_suffix}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Rezago (semanas)')
    ax.set_ylabel('Autocorrelación')

fig.suptitle('Firma Visual de la Estacionalidad en la ACF',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'ACF en lag 52  — Con estacionalidad: {acf_con[52]:.3f} | Sin: {acf_sin[52]:.3f}')
print(f'ACF en lag 104 — Con estacionalidad: {acf_con[104]:.3f} | Sin: {acf_sin[104]:.3f}')
print('\n→ Eliminar la estacionalidad (Video 3) es necesario antes de identificar p y q.')

---
## 7. Tabla de decisión: identificando p, d, q

Esta es la herramienta más práctica del video.  
Combinando los patrones de ACF y PACF podemos proponer el modelo ARIMA:

| ACF | PACF | Modelo sugerido |
|---|---|---|
| Decaimiento exponencial | Corte en lag **p** | **AR(p)** → ARIMA(p,d,0) |
| Corte en lag **q** | Decaimiento exponencial | **MA(q)** → ARIMA(0,d,q) |
| Decaimiento exponencial | Decaimiento exponencial | **ARMA(p,q)** → ARIMA(p,d,q) |
| Sin picos significativos | Sin picos significativos | **Ruido blanco** — ya es estacionaria |

> ⚠️ Esta tabla es una guía, no una regla absoluta. En la práctica se proponen 2-3 candidatos y se elige por criterios de información (AIC, BIC) — lo veremos en el Video de ARIMA.

In [ ]:
# Análisis automático de patrones en ACF y PACF de la serie preparada para ARIMA
# (desestacionalizada + diferenciada)
serie_arima = diff1.dropna()

acf_arima  = acf( serie_arima, nlags=20, fft=True)
pacf_arima = pacf(serie_arima, nlags=20, method='ywm')
conf_a     = 1.96 / np.sqrt(len(serie_arima))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_acf( serie_arima, lags=20, ax=axes[0], color=BLUE,   alpha=0.05)
plot_pacf(serie_arima, lags=20, ax=axes[1], color=PURPLE, alpha=0.05, method='ywm')

axes[0].set_title('ACF — Serie Lista para ARIMA\n(desestacionalizada + Δ1)',
                  fontsize=12, fontweight='bold')
axes[1].set_title('PACF — Serie Lista para ARIMA\n(desestacionalizada + Δ1)',
                  fontsize=12, fontweight='bold')

for ax in axes:
    ax.set_xlabel('Rezago')

fig.suptitle('Identificación de Parámetros ARIMA(p, d, q)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Detección automática de candidatos p y q
sig_acf_lags  = [i for i, v in enumerate(acf_arima)  if abs(v) > conf_a and i > 0]
sig_pacf_lags = [i for i, v in enumerate(pacf_arima) if abs(v) > conf_a and i > 0]

q_candidate = sig_acf_lags[0]  if sig_acf_lags  else 0
p_candidate = sig_pacf_lags[0] if sig_pacf_lags else 0

print('─' * 54)
print('  LECTURA DE CORRELOGRAMAS')
print('─' * 54)
print('  Serie analizada   : desestacionalizada + Δ1')
print('  d identificado    : 1  (Video 2 — prueba ADF/KPSS)')
print(f'  Rezagos ACF  significativos: {sig_acf_lags[:8]}')
print(f'  Rezagos PACF significativos: {sig_pacf_lags[:8]}')
print(f'  q candidato (ACF)  : {q_candidate}  → último lag ACF significativo antes del corte')
print(f'  p candidato (PACF) : {p_candidate}  → último lag PACF significativo antes del corte')
print('─' * 54)
print(f'  Modelo sugerido   : ARIMA({p_candidate}, 1, {q_candidate})')
print('  Modelos alternativos a comparar:')
print(f'    ARIMA({p_candidate}, 1, 0)  → puro AR')
print(f'    ARIMA(0, 1, {q_candidate})  → puro MA')
print(f'    ARIMA({p_candidate}, 1, {q_candidate})  → mixto')
print('─' * 54)
print('  En el Video de ARIMA elegiremos entre candidatos')
print('  usando AIC y BIC — no solo los correlogramas.')

---
## 8. Conclusiones

| Análisis | Hallazgo |
|---|---|
| ACF serie original | Decaimiento lento → tendencia | 
| ACF desestacionalizada | Picos periódicos desaparecen → estacionalidad removida |
| ACF lag 52 | Alto en serie original, bajo en desestacionalizada |
| PACF serie original | Primer lag dominante → fuerte proceso AR |
| Serie preparada (Δ1 + deseason) | Correlograma limpio → identificación de p y q posible |
| Modelo sugerido | ARIMA(p, 1, q) sobre serie desestacionalizada |

### El orden correcto de preparación para ARIMA

```
Serie original
    ↓  1. Desestacionalizar  (Video 3 — STL)
    ↓  2. Diferenciar d=1    (Video 2 — ADF/KPSS)
    ↓  3. Leer ACF → q
    ↓  4. Leer PACF → p
    ↓  5. Proponer ARIMA(p, 1, q)
    ↓  6. Elegir con AIC/BIC  (Video de ARIMA)
```

---

### Próximo video
**Video 5 — Detección de Outliers:** ¿Fue un error de datos o una promoción?  
Usaremos los residuales de la descomposición STL para identificar semanas anómalas  
y distinguir entre errores reales y eventos de negocio legítimos.